# DEXTRA · 10 — The dataset table your paper is missing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/10_dataset_table.ipynb)

Notebook 09 produced every **results** asset. But a conference reviewer reads Section III
first, and Section III needs a table describing the data itself: which cities, over what
period, how complete, and how often each city is actually dirty.

This notebook builds that table — your paper's **Table I** — straight from the dataset,
plus two figures that document coverage and severity honestly.

**It trains nothing.** No GPU, no HuggingFace token, no secrets — the dataset is public.
It runs in about a minute and can be re-run as often as you like.

One thing it is careful about: the severe-day counts here are over the **whole record**,
while notebook 09's per-city table counts severe targets inside the **held-out evaluation
window**. Those are different quantities and the captions say so, otherwise the two tables
look like they contradict each other.

In [ ]:
# ── setup ───────────────────────────────────────────────────────────────────
import importlib
import subprocess
import sys

for mod, spec in [("pandas", "pandas"), ("matplotlib", "matplotlib"),
                  ("numpy", "numpy"), ("pyarrow", "pyarrow")]:
    try:
        importlib.import_module(mod)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)

import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if not IN_COLAB:
    matplotlib.use("Agg")

REPO = "Aduitya9/Aduitya_dextra-aqi-india"
ROOT = f"https://huggingface.co/datasets/{REPO}/resolve/main"
OUT = Path("dataset_assets")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)

SEVERE_LINE = 200.0          # CPCB "very poor" boundary, the project's hazard line

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.autolayout": True,
})
C_FINAL, C_BASE, C_REF, C_GREY = "#0072B2", "#D55E00", "#009E73", "#999999"
print(f"pandas {pd.__version__} | matplotlib {matplotlib.__version__} | colab={IN_COLAB}")


def fetch(path, timeout=180):
    """Download one file from the public dataset and return its bytes."""
    with urllib.request.urlopen(f"{ROOT}/{path}", timeout=timeout) as r:
        return r.read()


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / "figures" / f"{name}.{ext}", bbox_inches="tight")
    print(f"  wrote figures/{name}.png + .pdf")
    if IN_COLAB:
        try:
            from IPython.display import display
            display(fig)
        except Exception:
            pass
    plt.close(fig)


def tex_escape(s):
    s = str(s)
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
                 ("#", r"\#"), ("_", r"\_"), ("{", r"\{"), ("}", r"\}"),
                 ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}")]:
        s = s.replace(a, b)
    return s


def to_tex(df, name, caption, label, floatfmt="%.1f", bold_rows=(), wide=False):
    """Write a booktabs table. Whole-number columns print as integers, not 2557.0,
    and text columns are left-aligned. `wide=True` emits a table* so a table with many
    columns spans both columns of an IEEEtran page instead of overflowing one."""
    d = df.copy()
    align = ""
    for c in d.columns:
        if pd.api.types.is_numeric_dtype(d[c]):
            finite = d[c].dropna()
            whole = len(finite) > 0 and bool((finite == finite.round()).all())
            fmt = "%.0f" if whole else floatfmt
            d[c] = d[c].map(lambda v, f=fmt: "" if pd.isna(v) else f % v)
            align += "r"
        else:
            align += "l"
    head = " & ".join(tex_escape(c) for c in d.columns) + r" \\"
    lines = []
    for _, r in d.iterrows():
        cells = [tex_escape(v) for v in r]
        if str(r.iloc[0]) in bold_rows:
            cells = [r"\textbf{" + c + "}" for c in cells]
        lines.append(" & ".join(cells) + r" \\")
    env = "table*" if wide else "table"
    tex = "\n".join([
        r"\begin{" + env + "}[t]", r"\centering",
        r"\caption{" + caption + "}", r"\label{" + label + "}",
        r"\footnotesize" if wide else r"",
        r"\begin{tabular}{" + align + "}", r"\toprule",
        head, r"\midrule", *lines, r"\bottomrule", r"\end{tabular}",
        r"\end{" + env + "}", ""])
    (OUT / "tables" / f"{name}.tex").write_text(tex, encoding="utf-8")
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
    print(f"  wrote tables/{name}.tex (+ .csv)")

In [ ]:
# ── fetch the daily table and the city metadata ─────────────────────────────
print("downloading the daily feature table (about 3.5 MB)...")
daily = pd.read_parquet(io.BytesIO(fetch("processed/city_daily_features.parquet")))
meta = pd.read_csv(io.BytesIO(fetch("meta/cities.csv")))

daily["date"] = pd.to_datetime(daily["date"])
print(f"  daily  {daily.shape[0]:,} rows x {daily.shape[1]} columns, "
      f"{daily['city'].nunique()} cities")
print(f"  meta   {meta.shape[0]} cities")
print(f"  record {daily['date'].min().date()} -> {daily['date'].max().date()}")

for need in ("city", "date", "AQI", "split"):
    assert need in daily.columns, f"daily table is missing {need!r}"

## Table I — the dataset, city by city

This is the table Section III needs. Every column is a plain fact about the data, not a
result: the record length, how complete the AQI series is, how many stations report on a
typical day, and how often the city is above the CPCB "very poor" line of 200.

The last column is the one that explains a result later in the paper: a model can only be
scored on severe-day ranking in cities that *have* severe days. Two cities in this study
essentially never cross the line, which is why the per-city results table is shorter than
this one.

In [ ]:
rows = []
for city in sorted(daily["city"].unique()):
    p = daily[daily["city"] == city].sort_values("date")
    aqi = p["AQI"]
    severe = aqi > SEVERE_LINE
    # an onset is the first day of a run above the line
    onsets = int((severe & ~severe.shift(1, fill_value=False)).sum())
    rows.append({
        "city": city,
        "days": int(len(p)),
        "AQI days (%)": float(100 * aqi.notna().mean()),
        "stations/day": float(p["n_stations"].median()) if "n_stations" in p else np.nan,
        "max stations": float(p["n_stations"].max()) if "n_stations" in p else np.nan,
        "mean AQI": float(aqi.mean()),
        "95th pct": float(aqi.quantile(0.95)),
        "severe days": int(severe.sum()),
        "severe (%)": float(100 * severe.mean()),
        "episodes": onsets,
    })
table1 = pd.DataFrame(rows)

# Only `state` and `archetype` are taken from the metadata. The metadata also lists a
# nominal CAAQMS station count per city (40 for Delhi), but the daily table never contains
# more than six reporting stations for any city, so the two numbers describe different
# things. Reporting the nominal count beside the observed one would invite a question the
# paper cannot answer, so Table I reports only what is actually in the data.
keep_meta = [c for c in ["city", "state", "archetype"] if c in meta.columns]
table1 = table1.merge(meta[keep_meta], on="city", how="left")
order = ["city", "state", "archetype", "days", "AQI days (%)",
         "stations/day", "max stations", "mean AQI", "95th pct",
         "severe days", "severe (%)", "episodes"]
table1 = table1[[c for c in order if c in table1.columns]]
table1 = table1.sort_values("severe days", ascending=False).reset_index(drop=True)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
print(table1.round(1).to_string(index=False))

span = f"{daily['date'].min().date()} to {daily['date'].max().date()}"
to_tex(table1, "table6_dataset",
       "The fifteen study cities, " + span + ". Severe days are those with a daily "
       "national AQI above 200 (the CPCB ``very poor'' boundary), counted over the whole "
       "record; an episode is a maximal run of consecutive severe days. Station counts are "
       "the stations reporting on a day in the assembled archive, not the full CAAQMS "
       "network. These counts describe the data, not the evaluation: the per-city results "
       "table counts severe targets inside the held-out window only, so its counts are "
       "smaller.",
       "tab:dataset", floatfmt="%.1f", wide=True)

## Table II — the leave-one-city-out protocol

This is the protocol behind the headline results, and it is the one Section III must
describe. It is **not** the `split` column that ships with the dataset: that column
defines a fixed seen/unseen split used by the earlier baseline notebooks, and reporting
it beside the leave-one-city-out results would describe the wrong experiment.

The rule, taken from `leave_one_out_split` in `src/data/assemble.py`, is the same in
every one of the fifteen folds:

| role | cities | period |
|---|---|---|
| train | the other fourteen | 2019-01-01 to 2023-12-31 |
| val — early stopping and alarm levels | the other fourteen | 2024 |
| test | **the held-out city only** | 2024-01-01 to 2025-12-31 |
| unused | the held-out city's own earlier years, and the other cities' 2025 | |

The held-out city contributes nothing to training, to validation, to the gap-filling
medians or to the alarm levels. Its own history reaches a forecast only as the input a
forecaster would genuinely have on the day. Training ends before the test period begins,
so the holdout is by city **and** by time.

In [ ]:
TRAIN_END = pd.Timestamp("2023-12-31")      # config.TRAIN_END_DATE
VAL_END = pd.Timestamp("2024-12-31")        # config.VAL_END_DATE

rows = []
for held in sorted(daily["city"].unique()):
    others = daily["city"] != held
    tr = daily[others & (daily["date"] <= TRAIN_END)]
    va = daily[others & (daily["date"] > TRAIN_END) & (daily["date"] <= VAL_END)]
    te = daily[~others & (daily["date"] > TRAIN_END)]
    rows.append({
        "held-out city": held,
        "train days": int(len(tr)),
        "val days": int(len(va)),
        "test days": int(len(te)),
        "test severe": int((te["AQI"] > SEVERE_LINE).sum()),
        "test severe (%)": float(100 * (te["AQI"] > SEVERE_LINE).mean()),
    })
table2 = pd.DataFrame(rows).sort_values("test severe", ascending=False).reset_index(drop=True)
print(f"fifteen folds | train <= {TRAIN_END.date()} | val {VAL_END.year} | "
      f"test {TRAIN_END.year + 1}-{VAL_END.year + 1}, held-out city only")
print(table2.round(1).to_string(index=False))

to_tex(table2, "table7_loco_protocol",
       "The leave-one-city-out protocol. In each of the fifteen folds the other fourteen "
       "cities train on 2019--2023 and validate on 2024, and the held-out city alone is "
       "scored on 2024--2025; it contributes nothing to training, validation, gap-filling "
       "medians or alarm levels. Severe days are days above AQI 200 in the held-out "
       "period. The per-city results table instead counts severe forecast \\emph{targets} "
       "in the complete-case evaluation frame, where each day appears once per horizon, so "
       "its counts differ by construction.",
       "tab:loco", floatfmt="%.1f")

## Figure A — what each city's air actually looks like

The spread of daily AQI per city, with the severe line drawn across it. This is the figure
that makes the per-city results honest: the cities on the right of this plot are the only
ones where severe-day ranking can be measured at all.

In [ ]:
# ordered by how often the city is severe, which is what the colour encodes and what
# Table I is sorted by, so the figure and the table tell the same story in the same order
ordered = table1.sort_values("severe (%)")["city"].tolist()
data = [daily.loc[daily["city"] == c, "AQI"].dropna().to_numpy() for c in ordered]

fig, ax = plt.subplots(figsize=(7.6, 4.4))
bp = ax.boxplot(data, vert=True, showfliers=False, patch_artist=True, widths=0.6)
for patch, city in zip(bp["boxes"], ordered):
    sev = float(table1.loc[table1["city"] == city, "severe (%)"].iloc[0])
    patch.set_facecolor(C_BASE if sev >= 10 else C_GREY)
    patch.set_alpha(0.85)
for part in ("medians", "whiskers", "caps"):
    for line in bp[part]:
        line.set_color("#333333")
ax.axhline(SEVERE_LINE, color=C_FINAL, lw=1.6, ls="--", zorder=5)
ax.text(0.4, SEVERE_LINE + 12, "severe (AQI 200)", color=C_FINAL, fontsize=8.5,
        fontweight="bold", va="bottom")
# set tick labels the version-safe way (no `labels=` / `tick_labels=` keyword)
ax.set_xticks(range(1, len(ordered) + 1))
ax.set_xticklabels(ordered, rotation=45, ha="right")
ax.set_ylabel("daily national AQI")
ax.set_title("Daily AQI by city, 2019-2025\n"
             "orange = at least one day in ten is severe; grey = rarely or never severe")
ax.set_ylim(0, 480)
save_fig(fig, "figA_city_aqi")

## Figure B — data coverage

The share of days in each year for which a city has an AQI value. Gaps are a fact of CPCB
monitoring and a reviewer will ask; showing them is cheaper than being asked.

In [ ]:
daily["year"] = daily["date"].dt.year
years = sorted(daily["year"].unique())
cities_by_sev = table1["city"].tolist()
grid = np.full((len(cities_by_sev), len(years)), np.nan)
for i, city in enumerate(cities_by_sev):
    for j, yr in enumerate(years):
        p = daily[(daily["city"] == city) & (daily["year"] == yr)]
        if len(p):
            grid[i, j] = 100 * p["AQI"].notna().mean()

fig, ax = plt.subplots(figsize=(6.4, 5.2))
# span the real range, so a genuinely bad year is not clipped to look like a mediocre one
vmin = float(np.floor(np.nanmin(grid) / 5.0) * 5.0) if np.isfinite(grid).any() else 0.0
im = ax.imshow(grid, aspect="auto", cmap="YlGnBu", vmin=vmin, vmax=100)
ax.set_xticks(range(len(years)))
ax.set_xticklabels(years)
ax.set_yticks(range(len(cities_by_sev)))
ax.set_yticklabels(cities_by_sev)
ax.grid(False)
for i in range(len(cities_by_sev)):
    for j in range(len(years)):
        if np.isfinite(grid[i, j]):
            dark = grid[i, j] > vmin + 0.65 * (100 - vmin)
            ax.text(j, i, f"{grid[i, j]:.0f}", ha="center", va="center", fontsize=7.5,
                    color="white" if dark else "#222222")
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cb.set_label("% of days with an AQI value")
ax.set_title("AQI coverage by city and year\n(cities ordered by severe-day count)")
save_fig(fig, "figB_coverage")

## A few sentences you can paste into Section III

Generated from the numbers above so they cannot drift from the table.

In [ ]:
n_cities = len(table1)
n_days = int(table1["days"].sum())
n_sev = int(table1["severe days"].sum())
cov = float(table1["AQI days (%)"].mean())
worst = table1.iloc[0]
clean = table1[table1["severe (%)"] < 1.0]["city"].tolist()

paragraph = (
    f"The study covers {n_cities} Indian cities over {span}, a total of {n_days:,} "
    f"city-days, of which {n_sev:,} ({100 * n_sev / n_days:.1f}%) exceed the CPCB "
    f"\"very poor\" boundary of AQI 200. Mean AQI coverage is {cov:.1f}% of days. "
    f"Severity is highly uneven: {worst['city']} is severe on {worst['severe (%)']:.1f}% "
    f"of days while "
    + (", ".join(clean[:-1]) + " and " + clean[-1] if len(clean) > 1 else
       (clean[0] if clean else "no city"))
    + (" are" if len(clean) != 1 else " is")
    + " severe on under 1% of days. Average precision for severe-day ranking is therefore "
      "reported only for the cities with enough severe targets to estimate it."
)
print(paragraph)
(OUT / "tables" / "section3_paragraph.txt").write_text(paragraph, encoding="utf-8")
print("\n  wrote tables/section3_paragraph.txt")

In [ ]:
# ── bundle everything ───────────────────────────────────────────────────────
zip_path = Path("dextra_dataset_assets.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
n_fig = len(list((OUT / "figures").glob("*.pdf")))
n_tab = len(list((OUT / "tables").glob("*.tex")))
print(f"\n{n_fig} figures (PNG + PDF) and {n_tab} LaTeX tables")
print(f"bundled into {zip_path} ({zip_path.stat().st_size / 1e3:.0f} KB)")

if IN_COLAB:
    print("\nDownloading the zip to your computer...")
    colab_files.download(str(zip_path))
else:
    print(f"\nnot in Colab; the bundle is at {zip_path.resolve()}")

print("\nThese are additional to notebook 09's bundle - different file names, "
      "nothing is overwritten.")